# Cat vs Dog Image Classification

This notebook trains a binary image classification model to distinguish between cats and dogs.

We use **transfer learning** with **InceptionV3**, a model pretrained on ImageNet.

### Workflow

1. Import libraries and configure the project
2. Load the pretrained InceptionV3 model
3. Build and compile the classification model
4. Load and preprocess the dataset
5. Train the model
6. Visualize the training results
7. Save the trained model

In [1]:
# Imports and configuration

import os

import keras
import tensorflow as tf
import matplotlib.pyplot as plt

from keras.applications.inception_v3 import InceptionV3, preprocess_input
from keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator


BASE_DIR = "data"

TRAIN_DIR = os.path.join(BASE_DIR, "train")
VALIDATION_DIR = os.path.join(BASE_DIR, "validation")

IMAGE_SIZE = (299, 299)
BATCH_SIZE = 20
EPOCHS = 1

## 1. Build the Model

We use InceptionV3 as the pretrained base model.

The original ImageNet classification layers are removed using `include_top=False`.

The pretrained layers are frozen, and a new classification head is added for our binary cat-vs-dog classification task.

The final layer contains one neuron with a sigmoid activation because we have two classes.

In [2]:
# Load the pretrained InceptionV3 model

base_model = InceptionV3(
    weights="imagenet",
    include_top=False,
    input_shape=(299, 299, 3)
)

base_model.trainable = False


# Add our classification head

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(1, activation="sigmoid")
])


# Compile the model

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ inception_v3 (Functional)       │ (None, 8, 8, 2048)     │    21,802,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,065,185 (84.17 MB)

 Trainable params: 262,401 (1.00 MB)

 Non-trainable params: 21,802,784 (83.17 MB)

## 2. Load and Preprocess the Dataset

The images are stored in separate folders for cats and dogs.

Because we are using InceptionV3, we use its `preprocess_input` function to preprocess the images.

The dataset structure is:

    data/
    ├── train/
    │   ├── cats/
    │   └── dogs/
    │
    └── validation/
        ├── cats/
        └── dogs/

In [3]:
# Create data generators

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

validation_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)


# Load training images

train_data = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary"
)


# Load validation images

validation_data = validation_datagen.flow_from_directory(
    VALIDATION_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary"
)


print("Class indices:", train_data.class_indices)

Found 1999 images belonging to 2 classes.
Found 1000 images belonging to 2 classes.
Class indices: {'cats': 0, 'dogs': 1}


## 3. Train the Model

The pretrained InceptionV3 layers are frozen, so only the new classification layers are trained.

In [4]:
history = model.fit(
    train_data,
    validation_data=validation_data,
    epochs=EPOCHS
)

100/100 ━━━━━━━━━━━━━━━━━━━━ 116s 1s/step - accuracy: 0.9755 - loss: 0.0784 - val_accuracy: 0.9910 - val_loss: 0.0234


##  Save the Trained Model

The trained model is saved so that we can use it later for prediction without training it again.

In [7]:
os.makedirs("models", exist_ok=True)

model.save("models/cat_dog_inceptionv3.keras")

print("Model saved successfully.")

Model saved successfully.
